In [1]:
%pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

variables = pd.read_parquet("data/clean/variables_modelo.parquet")
X = variables.drop(columns=["id_cliente", "sexo", "impago_mes_siguiente"])
y = variables["impago_mes_siguiente"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_entreno, X_valid, y_entreno, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

In [5]:
modelo_lgb = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=15,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
modelo_lgb.fit(
    X_entreno, y_entreno,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
print("Árboles usados:", modelo_lgb.best_iteration_)

/opt/miniconda3/envs/credito/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Árboles usados: 335


In [6]:
modelo_log = joblib.load("data/clean/modelo_logistico.joblib")

def gini(modelo, X_, y_):
    return 2 * roc_auc_score(y_, modelo.predict_proba(X_)[:, 1]) - 1

comparacion = pd.DataFrame({
    "Gini entrenamiento": [gini(modelo_log, X_train, y_train), gini(modelo_lgb, X_train, y_train)],
    "Gini prueba": [gini(modelo_log, X_test, y_test), gini(modelo_lgb, X_test, y_test)],
}, index=["Regresión logística", "LightGBM"]).round(3)
comparacion["diferencia"] = (comparacion["Gini entrenamiento"] - comparacion["Gini prueba"]).round(3)
display(comparacion)

,Gini entrenamiento,Gini prueba,diferencia
Regresión logística,0.511,0.478,0.033
LightGBM,0.632,0.548,0.084


In [7]:
modelo_lgb = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=7,
    max_depth=4,
    min_child_samples=300,
    reg_lambda=5.0,
    subsample=0.7,
    subsample_freq=1,
    colsample_bytree=0.7,
    random_state=42,
    verbose=-1,
)
modelo_lgb.fit(
    X_entreno, y_entreno,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="auc",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
print("Árboles usados:", modelo_lgb.best_iteration_)

Árboles usados: 831


In [8]:
modelo_log = joblib.load("data/clean/modelo_logistico.joblib")

def gini(modelo, X_, y_):
    return 2 * roc_auc_score(y_, modelo.predict_proba(X_)[:, 1]) - 1

comparacion = pd.DataFrame({
    "Gini entrenamiento": [gini(modelo_log, X_train, y_train), gini(modelo_lgb, X_train, y_train)],
    "Gini prueba": [gini(modelo_log, X_test, y_test), gini(modelo_lgb, X_test, y_test)],
}, index=["Regresión logística", "LightGBM"]).round(3)
comparacion["diferencia"] = (comparacion["Gini entrenamiento"] - comparacion["Gini prueba"]).round(3)
display(comparacion)

,Gini entrenamiento,Gini prueba,diferencia
Regresión logística,0.511,0.478,0.033
LightGBM,0.614,0.547,0.067


In [9]:
importancia = pd.Series(
    modelo_lgb.booster_.feature_importance(importance_type="gain"), index=X.columns
)
importancia = (importancia / importancia.sum()).sort_values(ascending=False)
display(importancia.head(10).round(3).to_frame("parte de la importancia"))

joblib.dump(modelo_lgb, "data/clean/modelo_lightgbm.joblib")

,parte de la importancia
retraso_sep,0.350
meses_con_retraso,0.172
peor_retraso_6m,0.161
uso_medio_6m,0.072
uso_limite_sep,0.069
limite_credito,0.058
parte_pagada_6m,0.046
cambio_uso,0.042
edad,0.020
estado_civil_soltero,0.005


['data/clean/modelo_lightgbm.joblib']